In [199]:
import pandas as pd


stations_initial = pd.read_csv('../../data/initial_data/ps_stations_initial.csv', parse_dates=['created'])

In [200]:
stations_initial

,id,address,code,created,login,partner_name,employee_name
0,1,"121601, Московская обл, Москва, Филевский б-р, 6",CODE,2023-07-04 15:57:36.000000,ily,Beeline,Галина
1,2,"141310, Московская обл, Сергиево-Посадский р-н...",L422,2023-07-04 15:58:42.000000,station1,Связной,Галина
2,3,"142600, Московская обл, Орехово-Зуево г, Якова...",L382,2023-07-04 15:58:42.000000,station2,Связной,Галина
3,4,"142500, Московская обл, Павлово-Посадский р-н,...",L387,2023-07-04 15:58:43.000000,station3,Связной,Галина
4,5,"141031, Московская обл, Мытищинский р-н, Алтуф...",L350,2023-07-04 15:58:43.000000,station4,Связной,Галина
...,...,...,...,...,...,...,...
4059,4063,"г. Сочи, ул. Новая заря, дом 7, ТЦ МореМолл",RSTR_СОЧ_МОРЕМОЛЛ,2026-09-15 12:01:48.469836,station4264,Inventive Group,Галина
4060,4064,"г. Сочи, ул. Новая заря, дом 7, ТЦ МореМолл SA...",SMSG_СОЧ_МОРЕМОЛЛ,2026-09-15 12:05:45.030168,station4265,Inventive Group,Галина
4061,4065,"г. Котельники, 1-й Покровский проезд, д.1, ТЦ...",RSTR_МСК_МЕГА_БЕЛАЯ_ДАЧА,2026-09-15 12:13:14.727116,station4266,Inventive Group,Галина
4062,4066,"г. Краснодар, ул. Головатого, д.313,(пом.1-044...",SMSG_КДР_ГАЛЕРЕЯ,2026-09-15 20:44:28.208919,station4267,Inventive Group,Галина


In [201]:
stations_initial.info()

<class 'pandas.DataFrame'>
RangeIndex: 4064 entries, 0 to 4063
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             4064 non-null   int64         
 1   address        3726 non-null   str           
 2   code           4064 non-null   str           
 3   created        4064 non-null   datetime64[us]
 4   login          4064 non-null   str           
 5   partner_name   4064 non-null   str           
 6   employee_name  4064 non-null   str           
dtypes: datetime64[us](1), int64(1), str(5)
memory usage: 222.4 KB


In [202]:
import re


def city_address(address):

    largest_cities = [
    "Москва", "Санкт-Петербург", "Новосибирск", "Екатеринбург", "Казань",
    "Нижний Новгород", "Челябинск", "Красноярск", "Самара",
    "Омск", "Воронеж", "Пермь",
    "Саратов", "Тюмень",
    "Барнаул", "Иркутск", "Хабаровск", "Ярославль",
    "Владивосток", "Махачкала", "Томск", "Кемерово",
    "Рязань", "Пенза",
    "Липецк", "Тула", "Киров", "Калининград",
    "Брянск", "Курск", "Иваново", "Улан-Удэ",
    "Тверь", "Архангельск",
    "Московская область", "Московская обл"
]

    if isinstance(address, str):
        for city in largest_cities:
            if re.search(r'\bМосковская область\b', address) or re.search(r'\bМосковская обл\b', address):
                return 'Московская область'
            if re.search(rf'\b{city}(\b|,)', address):
                return city
    else:
        return 'Неизвестно'
    return 'Малый город'


In [203]:
stations_initial['cities'] = stations_initial['address'].apply(city_address)

In [204]:
list(stations_initial['cities'].value_counts().items())

[('Малый город', 1959),
 ('Московская область', 454),
 ('Неизвестно', 338),
 ('Москва', 249),
 ('Санкт-Петербург', 231),
 ('Новосибирск', 56),
 ('Екатеринбург', 50),
 ('Воронеж', 44),
 ('Тюмень', 43),
 ('Нижний Новгород', 39),
 ('Рязань', 37),
 ('Курск', 36),
 ('Ярославль', 33),
 ('Красноярск', 29),
 ('Самара', 28),
 ('Тула', 28),
 ('Омск', 26),
 ('Саратов', 25),
 ('Брянск', 25),
 ('Пермь', 24),
 ('Казань', 24),
 ('Липецк', 24),
 ('Хабаровск', 23),
 ('Иваново', 22),
 ('Калининград', 21),
 ('Кемерово', 21),
 ('Тверь', 20),
 ('Пенза', 20),
 ('Владивосток', 19),
 ('Томск', 19),
 ('Челябинск', 18),
 ('Барнаул', 17),
 ('Улан-Удэ', 16),
 ('Киров', 16),
 ('Архангельск', 15),
 ('Иркутск', 15)]

In [205]:
stations = stations_initial.drop(['address', 'code', 'login'], axis=1)
stations['id'].astype(int)

0          1
1          2
2          3
3          4
4          5
        ... 
4059    4063
4060    4064
4061    4065
4062    4066
4063    4067
Name: id, Length: 4064, dtype: int64

### В ходе исследования данных было выяснено, что некоторые станции не сделали ни одной продажи. Исключим эти станции из рассмотрения.

In [206]:
sales_initial = pd.read_csv('../../data/initial_data/ps_stats_initial.csv')
work_stations = set(map(int, set(sales_initial['station_id'].dropna())))
len(work_stations)

2463

In [207]:
stations_to_delete = {
    495, 1283, 1288, 1800, 1802, 1803, 1822, 1880, 1881, 1934,
    1936, 1947, 1950, 1951, 1952, 1954, 1971, 2224, 2242, 2243,
    2247, 2342, 2343, 2345, 2392, 2433, 2450, 2562, 2593, 2760,
    2902, 2903, 2904, 2910, 3267, 3268, 3270, 3300, 3353, 3354,
    3366, 3441, 3488, 3494, 3511, 3554, 3577, 3585, 3619, 3630,
    3636, 3637, 3638, 1227, 1538, 1539, 1540, 1805, 1824, 1826,
    1828, 1926, 1957, 1972, 2254, 3316, 3361, 3508, 3602, 1801, 1
}
work_stations = work_stations - stations_to_delete

print(f'Количество рабочих станций: {len(work_stations)}')

Количество рабочих станций: 2423


In [208]:
stations = stations[stations['id'].isin(work_stations)]

# Исключили несуществующие станции
Посмотрим на то, что у нас получилось

In [209]:
stations.reset_index(drop=True, inplace=True)
stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 2423 entries, 0 to 2422
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             2423 non-null   int64         
 1   created        2423 non-null   datetime64[us]
 2   partner_name   2423 non-null   str           
 3   employee_name  2423 non-null   str           
 4   cities         2423 non-null   str           
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 94.8 KB


In [211]:
stations['created'] = stations['created'].dt.date

In [212]:
stations

,id,created,partner_name,employee_name,cities
0,277,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
1,279,2023-07-04,Мегафон,Галина,Малый город
2,280,2023-07-04,Мегафон,Екатерина,Малый город
3,281,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
4,283,2023-07-04,Мегафон,Роман,Московская область
...,...,...,...,...,...
2418,4055,2026-09-11,Inventive Group,Никита,Москва
2419,4056,2026-09-14,Inventive Group,Екатерина,Санкт-Петербург
2420,4057,2026-09-14,Inventive Group,Екатерина,Санкт-Петербург
2421,4063,2026-09-15,Inventive Group,Галина,Малый город
